## P1

In [1]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
current_dir

WindowsPath('E:/Openai/abtest/notebooks')

In [2]:
project_root = current_dir.parent
data_path = project_root / "data" / "raw" / "hillstrom.csv"

print(project_root)
print(data_path)
print(data_path.exists())

E:\Openai\abtest
E:\Openai\abtest\data\raw\hillstrom.csv
True


In [3]:
df = pd.read_csv(data_path)

print(type(df))
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
(64000, 12)


In [4]:
print(df.columns.tolist())
df.head()

['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


In [6]:
missing_counts = df.isna().sum()
missing_counts

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

In [7]:
df["segment"].value_counts(dropna=False)

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [8]:
df["visit"].value_counts(dropna=False)

visit
0    54606
1     9394
Name: count, dtype: int64

In [9]:
df["conversion"].value_counts(dropna=False)

conversion
0    63422
1      578
Name: count, dtype: int64

In [10]:
zero_spend_count = (df["spend"] == 0).sum()
negative_spend_count = (df["spend"] < 0).sum()

print("零消费人数：", zero_spend_count)
print("负金额记录数：", negative_spend_count)

零消费人数： 63422
负金额记录数： 0


In [11]:
purchase_mismatch = (df["conversion"] == 1) != (df["spend"] > 0)

print("购买标记与消费金额不一致的记录数：", purchase_mismatch.sum())

购买标记与消费金额不一致的记录数： 0


In [12]:
purchase_without_visit = (df["conversion"] == 1) & (df["visit"] == 0)

print("购买但未访问的记录数：", purchase_without_visit.sum())

购买但未访问的记录数： 0


In [13]:
df[["recency", "history", "spend"]].describe()

,recency,history,spend
count,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,1.050908
std,3.507592,256.158608,15.036448
min,1.000000,29.990000,0.000000
25%,2.000000,64.660000,0.000000
50%,6.000000,158.110000,0.000000
75%,9.000000,325.657500,0.000000
max,12.000000,3345.930000,499.000000


In [14]:
zero_spend_rate = zero_spend_count / len(df)
print(f"零消费客户占比：{zero_spend_rate:.2%}")

零消费客户占比：99.10%


1. 数据有多少行、多少列，分为哪三组；
2. 零消费比例是多少，为什么零消费不等于缺失；
3. 两项逐行一致性检查发现了什么，以及这些检查尚不能证明什么。



1. 数据共有 64,000 行、12 列。按照 `segment` 分为 `Womens E-Mail`、`Mens E-Mail` 和 `No E-Mail` 三个实验组。
2. 零消费客户有 63,422 人，占全部客户的 99.10%。`spend=0` 是已记录的消费金额，表示该客户在观察期内没有消费，不属于缺失值；这些客户仍然要保留在总体人数和人均消费的分母中。
3. 购买标记与消费金额不一致的记录数为 0，购买但未访问的记录数也为 0，说明 `conversion`、`spend` 和 `visit` 在这两项规则下逐行一致。这些检查只能验证相关字段的内部对应关系，不能证明三组实验前客户特征已经平衡，也不能据此判断邮件是否产生了效果。
P

## P2
业务问题是：三种邮件策略分别覆盖多少客户、带来多少访问者和购买者、产生多少收入，以及各组的访问率、购买率和人均收入。所有指标的分母都必须包含组内全部客户，不能只保留访问者或购买者。

 第一个小任务：用 Pandas 复现三组 SQL 汇总
1. `groupby("segment")` 做了什么？
2. `.size()` 统计的是什么？
3. 三组人数相加为什么应当等于 64,000？

1. `groupby("segment")` 按照 `segment` 的不同取值把数据分组，为后续分别计算每组指标做准备；此时还没有执行具体统计。
2. `.size()` 统计每个组包含的行数。由于本数据一行代表一名客户，因此这里表示每组客户数。
3. `segment` 没有缺失值，并且每行只属于一个实验组，所以三个组完整且不重复地覆盖全部 64,000 行，三组人数之和应等于 64,000。

In [15]:
group_sizes = df.groupby("segment").size()
group_sizes

segment
Mens E-Mail      21307
No E-Mail        21306
Womens E-Mail    21387
dtype: int64

In [16]:
print("三组人数之和：", group_sizes.sum())
print("数据总行数：", len(df))
print("是否相等：", group_sizes.sum() == len(df))

三组人数之和： 64000
数据总行数： 64000
是否相等： True


第二个小任务：汇总人数和收入

1. 为什么对 `visit` 求和可以得到访问人数？
2. 为什么对 `conversion` 求和可以得到购买人数？
3. 对 `spend` 求和得到的是什么？
4. 计算这些指标前为什么不能筛选 `visit == 1` 或 `conversion == 1`？

1. `visit` 是 0/1 标记，对每组求和时，0 不增加结果，1 增加 1，因此总和就是每组访问人数。
2. `conversion` 同样是 0/1 标记，所以每组求和就是每组购买人数。
3. 对 `spend` 按组求和，得到每个实验组全部客户在观察期内的总消费金额，也就是该组总收入。
4. 不能提前筛选 `visit == 1` 或 `conversion == 1`，因为这样会删除未访问或未购买的客户，改变原本随机分组的全部客户分母。筛选后计算的购买率、访问率和人均收入会变成条件指标，不能表示每个实验组全部客户的结果。例如只保留购买客户后，购买率会变成 100%，平均消费也会变成购买客户的平均消费，而不是全部客户的人均收入。

In [17]:
group_totals = (
    df.groupby("segment")
      .agg(
          visitors=("visit", "sum"),
          buyers=("conversion", "sum"),
          revenue=("spend", "sum"),
      )
)

group_totals

,visitors,buyers,revenue
segment,,,
Mens E-Mail,3894,267,30311.69
No E-Mail,2262,122,13908.33
Womens E-Mail,3238,189,23038.11


P2 第三个小任务：计算比例与人均收入

1. 访问率的分子和分母分别是什么？
2. 购买率的分子和分母分别是什么？
3. 人均收入的分子和分母分别是什么？
4. 为什么人均收入的分母要包含 `spend=0` 的客户？

1. 访问率的分子是访问人数，分母是全部客户数
2. 购买率的分子是购买人数，分母是全部客户数
3. 人均收入分子是总收入，分母是全部客户数
4. 因为我们要评估邮件策略对全部被分配客户的平均收入贡献。没有消费也是策略实施后观察到的结果，因此这些客户要计入分母。删除他们后，得到的就会是购买客户的平均消费金额，回答了另一个问题。

In [18]:
group_summary = group_totals.copy()
group_summary["customers"] = group_sizes

group_summary = group_summary[
    ["customers", "visitors", "buyers", "revenue"]
]

group_summary

,customers,visitors,buyers,revenue
segment,,,,
Mens E-Mail,21307,3894,267,30311.69
No E-Mail,21306,2262,122,13908.33
Womens E-Mail,21387,3238,189,23038.11


In [19]:
group_summary["visit_rate"] = (
    group_summary["visitors"] / group_summary["customers"]
)

group_summary["conversion_rate"] = (
    group_summary["buyers"] / group_summary["customers"]
)

group_summary["revenue_per_customer"] = (
    group_summary["revenue"] / group_summary["customers"]
)

group_summary

,customers,visitors,buyers,revenue,visit_rate,conversion_rate,revenue_per_customer
segment,,,,,,,
Mens E-Mail,21307,3894,267,30311.69,0.182757,0.012531,1.422617
No E-Mail,21306,2262,122,13908.33,0.106167,0.005726,0.652789
Womens E-Mail,21387,3238,189,23038.11,0.151400,0.008837,1.077202


P2 第四个小任务：用平均值核对人均收入

1. 为什么组内 `spend` 的平均值应等于总收入除以全部客户数？
2. 如果 `spend` 存在缺失值，直接使用 `.mean()` 时，分母还一定是全部客户数吗？

1. 因为spend算的是全部人的人均收入，其中包括了消费的和没有消费的人
2. Pandas 的 .mean() 默认跳过缺失值，但会保留数字0. 分母将缺失的客户忽略了

In [20]:
mean_spend = df.groupby("segment")["spend"].mean()

revenue_check = pd.DataFrame({
    "总收入除以客户数": group_summary["revenue_per_customer"],
    "直接求平均值": mean_spend,
})

revenue_check["差值"] = (
    revenue_check["总收入除以客户数"]
    - revenue_check["直接求平均值"]
)

revenue_check

,总收入除以客户数,直接求平均值,差值
segment,,,
Mens E-Mail,1.422617,1.422617,0.0
No E-Mail,0.652789,0.652789,0.0
Womens E-Mail,1.077202,1.077202,0.0


P2 第五个小任务：用平均值核对访问率和购买率

1. 为什么 `visit` 的平均值等于访问率？
2. 为什么 `conversion` 的平均值等于购买率？
3. 要让这两个平均值表示“全部客户中的比例”，数据需要满足什么条件？

1. 因为visit是用0/1来标记是否访问的，所以对visit求平均值和求访问率的公式是一样的
2. 同理于visit
3. visit 和 conversion 只取0或1、没有缺失，并且计算时保留每组全部随机分配的客户，没有提前筛选访问者或购买者。

In [21]:
mean_visit = df.groupby("segment")["visit"].mean()
mean_conversion = df.groupby("segment")["conversion"].mean()

print("按组求平均值得到的访问率：")
print(mean_visit)

print("按组求平均值得到的购买率：")
print(mean_conversion)

按组求平均值得到的访问率：
segment
Mens E-Mail      0.182757
No E-Mail        0.106167
Womens E-Mail    0.151400
Name: visit, dtype: float64
按组求平均值得到的购买率：
segment
Mens E-Mail      0.012531
No E-Mail        0.005726
Womens E-Mail    0.008837
Name: conversion, dtype: float64


In [22]:
rate_check = pd.DataFrame({
    "人数计算的访问率": group_summary["visit_rate"],
    "平均值计算的访问率": mean_visit,
    "人数计算的购买率": group_summary["conversion_rate"],
    "平均值计算的购买率": mean_conversion,
})
rate_check["访问率差值"] = (
    rate_check["人数计算的访问率"]
    - rate_check["平均值计算的访问率"]
)

rate_check["购买率差值"] = (
    rate_check["人数计算的购买率"]
    - rate_check["平均值计算的购买率"]
)

rate_check

,人数计算的访问率,平均值计算的访问率,人数计算的购买率,平均值计算的购买率,访问率差值,购买率差值
segment,,,,,,
Mens E-Mail,0.182757,0.182757,0.012531,0.012531,0.0,0.0
No E-Mail,0.106167,0.106167,0.005726,0.005726,0.0,0.0
Womens E-Mail,0.151400,0.151400,0.008837,0.008837,0.0,0.0


In [23]:
import duckdb

sql_connection = duckdb.connect()
sql_connection.register("customers", df)

sql_path = project_root / "exercises" / "01_metrics.sql"
sql_query = sql_path.read_text(encoding="utf-8-sig")

sql_summary = sql_connection.execute(sql_query).df()
sql_summary

,segment,n,visitors,buyers,revenue,revenue_per_customer,visit_rate,conversion_rate,spend_variance
0,Mens E-Mail,21307,3894.0,267.0,30311.69,1.422617,0.182757,0.012531,315.211799
1,No E-Mail,21306,2262.0,122.0,13908.33,0.652789,0.106167,0.005726,134.286372
2,Womens E-Mail,21387,3238.0,189.0,23038.11,1.077202,0.151400,0.008837,228.496675


In [24]:
sql_aligned = (
    sql_summary
    .rename(columns={"n": "customers"})
    .set_index("segment")
)

sql_aligned = sql_aligned[group_summary.columns]

metric_difference = group_summary - sql_aligned
metric_difference

,customers,visitors,buyers,revenue,visit_rate,conversion_rate,revenue_per_customer
segment,,,,,,,
Mens E-Mail,0,0.0,0.0,-2.182787e-11,0.0,0.0,-1.110223e-15
No E-Mail,0,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00
Womens E-Mail,0,0.0,0.0,-3.637979e-12,0.0,0.0,-2.220446e-16


为什么比较 SQL 和 Pandas 结果前，要把实验组索引和指标列名统一？

统一实验组索引和指标列名，是为了让 Pandas 按名称匹配同一实验组、同一指标，再计算差值，从而检查 SQL 与 Pandas 的计算结果是否一致。

In [25]:
toy_df = pd.DataFrame({
    "segment": ["A", "A", "A", "B", "B", "B"],
    "visit": [0, 1, None, 1, 0, 0],
})

toy_df

print("每组全部行数：")
print(toy_df.groupby("segment").size())

print("每组 visit 的非缺失值数量：")
print(toy_df.groupby("segment")["visit"].count())

每组全部行数：
segment
A    3
B    3
dtype: int64
每组 visit 的非缺失值数量：
segment
A    2
B    3
Name: visit, dtype: int64


1. A 组的 .size() 和 visit 的 .count() 分别是多少？为什么不同？
2. visit=0 的记录会被 .count() 统计进去吗？为什么？

1. 3，2，count（）把缺失值忽略了
2. 会，0不算缺失值

P2 小变式：增加每组未访问人数

计算关系：未访问人数 = 全部客户数 − 访问人数。

In [26]:
group_summary["non_visitors"] = (
    group_summary["customers"] - group_summary["visitors"]
)

group_summary

,customers,visitors,buyers,revenue,visit_rate,conversion_rate,revenue_per_customer,non_visitors
segment,,,,,,,,
Mens E-Mail,21307,3894,267,30311.69,0.182757,0.012531,1.422617,17413
No E-Mail,21306,2262,122,13908.33,0.106167,0.005726,0.652789,19044
Womens E-Mail,21387,3238,189,23038.11,0.151400,0.008837,1.077202,18149


1. 每组的访问人数加未访问人数，应该等于哪一列？
2. 增加未访问人数这一列后，原来的访问率、购买率和人均收入的分母需要改变吗？为什么？

1. 全部客户数
2. 不需要改变，未访问数不影响原来指标的计算方式